# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Manjit-yadav/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

df.head()

Rows: 30000
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [3]:
df["age_bucket"] = pd.cut(
    df["content_age_days"],
    bins=[-1, 90, 180, 365, np.inf],
    labels=["0-90", "91-180", "181-365", "365+"]
)

age_check = (
    df.groupby("age_bucket", observed=True)
      .agg(
          n=("content_id", "count"),
          avg_ctr=("ctr", "mean"),
          avg_position=("avg_position", "mean"),
          avg_impressions=("impressions_90d", "mean")
      )
      .reset_index()
)

age_check

,age_bucket,n,avg_ctr,avg_position,avg_impressions
0,0-90,492,0.338577,16.155081,3209.445122
1,91-180,11780,0.345821,14.125611,5101.726401
2,181-365,11368,0.817498,16.145980,5398.772871
3,365+,6360,0.281184,20.813821,5182.445755


In [4]:
df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[0, 3, 5, 10, 20, np.inf],
    labels=["1-3", "4-5", "6-10", "11-20", "20+"]
)

position_check = (
    df.groupby("position_bucket", observed=True)
      .agg(
          n=("content_id", "count"),
          avg_ctr=("ctr", "mean"),
          avg_impressions=("impressions_90d", "mean")
      )
      .reset_index()
)

position_check

,position_bucket,n,avg_ctr,avg_impressions
0,1-3,1141,2.714303,6626.347940
1,4-5,2782,1.104820,11036.156722
2,6-10,9060,0.511708,6474.484768
3,11-20,7273,0.323443,3137.629589
4,20+,8539,0.211333,4247.178241


In [5]:
position_ctr_median = (
    df.groupby("position_bucket", observed=True)["ctr"]
      .median()
)

df["position_ctr_median"] = df["position_bucket"].map(position_ctr_median)

In [6]:
df["ctr_below_position_median"] = (
    df["ctr"] < df["position_ctr_median"]
)

In [7]:
df["age_points"] = np.select(
    [
        df["content_age_days"] >= 365,
        df["content_age_days"] >= 180
    ],
    [
        2,
        1
    ],
    default=0
)

df["ctr_points"] = np.where(
    df["ctr_below_position_median"],
    2,
    0
)

df["score"] = df["age_points"] + df["ctr_points"]

In [8]:
df["reason_code"] = np.select(
    [
        (df["age_points"] >= 2) & (df["ctr_points"] >= 2),
        (df["age_points"] >= 2),
        (df["ctr_points"] >= 2)
    ],
    [
        "STALE_AND_LOW_CTR",
        "STALE",
        "LOW_CTR_FOR_POSITION"
    ],
    default="NO_STRONG_SIGNAL"
)

In [9]:
df["action"] = np.where(
    df["score"] >= 2,
    "REVIEW_REFRESH",
    "LOW_PRIORITY"
)

In [10]:
baseline_queue = (
    df[
        [
            "content_id",
            "client_id",
            "score",
            "reason_code",
            "action",
            "content_age_days",
            "avg_position",
            "ctr",
            "position_bucket"
        ]
    ]
    .sort_values(
        ["score", "content_age_days"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

baseline_queue.head(10)

,content_id,client_id,score,reason_code,action,content_age_days,avg_position,ctr,position_bucket
0,content_5d64fc00babd,client_9400f1b21c,4,STALE_AND_LOW_CTR,REVIEW_REFRESH,564,7.3,0.00,6-10
1,content_9d94766abf35,client_9400f1b21c,4,STALE_AND_LOW_CTR,REVIEW_REFRESH,557,9.8,0.04,6-10
2,content_b6f1aab067b3,client_9400f1b21c,4,STALE_AND_LOW_CTR,REVIEW_REFRESH,557,11.3,0.08,11-20
3,content_8e90307c82dc,client_9400f1b21c,4,STALE_AND_LOW_CTR,REVIEW_REFRESH,557,10.9,0.00,11-20
4,content_7424684ce198,client_9400f1b21c,4,STALE_AND_LOW_CTR,REVIEW_REFRESH,557,15.5,0.00,11-20
5,content_a0c3be8b0794,client_9400f1b21c,4,STALE_AND_LOW_CTR,REVIEW_REFRESH,557,8.2,0.00,6-10
6,content_64b2508ad098,client_9400f1b21c,4,STALE_AND_LOW_CTR,REVIEW_REFRESH,557,9.2,0.07,6-10
7,content_c3b5e633104e,client_9400f1b21c,4,STALE_AND_LOW_CTR,REVIEW_REFRESH,557,8.0,0.09,6-10
8,content_cbb8dfc20844,client_9400f1b21c,4,STALE_AND_LOW_CTR,REVIEW_REFRESH,557,17.5,0.00,11-20
9,content_c79aa397ddea,client_9400f1b21c,4,STALE_AND_LOW_CTR,REVIEW_REFRESH,557,7.0,0.00,6-10


In [11]:
output_path = "/content/work/outputs/baseline_action_score.csv"

baseline_queue.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

OSError: Cannot save file into a non-existent directory: '/content/work/outputs'

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.